# Speaker-disjoint CV and M0/A/B1/B2/C1/C2 Ridge model fitting

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
REPORT_DIR = PROJECT / "reports"
SPLIT_DIR = PROJECT / "data/splits"
CHECK = PROJECT / "checkpoints/research_progress.json"

SPEAKER_CSV = REPORT_DIR / "step05_speaker_fluency_audit.csv"
FOLD_CSV = SPLIT_DIR / "speaker_groupkfold5_candidate.csv"
REPORT = REPORT_DIR / "step05_speaker_split_candidate_qc.json"

EXPECTED_SHA = (
    "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_SHA

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df["fluency"].notna().all()

# One record per audio; H1-H5 must not inflate counts.
audio = df[
    ["pilot_index", "audio_id", "speaker", "fluency"]
].drop_duplicates().sort_values(
    "pilot_index"
).reset_index(drop=True)

assert len(audio) == 400
assert audio["audio_id"].is_unique
assert audio.groupby("speaker")["fluency"].count().sum() == 400

# Speaker-level descriptive statistics.
speaker_stats = (
    audio.groupby("speaker")
    .agg(
        audio_count=("audio_id", "count"),
        mean_fluency=("fluency", "mean"),
        median_fluency=("fluency", "median"),
        sd_fluency=("fluency", "std"),
        min_fluency=("fluency", "min"),
        max_fluency=("fluency", "max")
    )
    .reset_index()
)

score_counts = pd.crosstab(
    audio["speaker"],
    audio["fluency"]
)

score_counts.columns = [
    f"score_{int(c)}_count"
    for c in score_counts.columns
]

speaker_stats = speaker_stats.merge(
    score_counts.reset_index(),
    on="speaker",
    validate="one_to_one"
)

# Candidate GroupKFold: validation speakers are unseen in training.
gkf = GroupKFold(n_splits=5)
fold_rows = []
assignments = []

for fold_number, (train_idx, val_idx) in enumerate(
    gkf.split(
        audio,
        y=audio["fluency"],
        groups=audio["speaker"]
    ),
    start=1
):
    train = audio.iloc[train_idx]
    val = audio.iloc[val_idx]

    train_speakers = set(train["speaker"])
    val_speakers = set(val["speaker"])

    assert train_speakers.isdisjoint(val_speakers)
    assert set(train["audio_id"]).isdisjoint(
        set(val["audio_id"])
    )

    fold_rows.append({
        "fold": fold_number,
        "train_speakers": len(train_speakers),
        "val_speakers": len(val_speakers),
        "train_audios": len(train),
        "val_audios": len(val),
        "train_mean": float(train["fluency"].mean()),
        "val_mean": float(val["fluency"].mean()),
        "train_sd": float(train["fluency"].std()),
        "val_sd": float(val["fluency"].std()),
        "train_min": int(train["fluency"].min()),
        "train_max": int(train["fluency"].max()),
        "val_min": int(val["fluency"].min()),
        "val_max": int(val["fluency"].max()),
        "val_speaker_ids": ",".join(sorted(val_speakers))
    })

    for _, record in val.iterrows():
        assignments.append({
            "pilot_index": int(record["pilot_index"]),
            "audio_id": str(record["audio_id"]),
            "speaker": str(record["speaker"]),
            "fluency": int(record["fluency"]),
            "validation_fold": fold_number
        })

fold_stats = pd.DataFrame(fold_rows)
assignment_df = pd.DataFrame(assignments).sort_values(
    "pilot_index"
).reset_index(drop=True)

assert len(assignment_df) == 400
assert assignment_df["audio_id"].is_unique
assert assignment_df["validation_fold"].between(1, 5).all()

# Every speaker must appear in exactly one validation fold.
assert assignment_df.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

# Every audio's five hypotheses must inherit the same fold.
check = df[
    ["audio_id", "speaker", "hypothesis_id"]
].merge(
    assignment_df[
        ["audio_id", "speaker", "validation_fold"]
    ],
    on=["audio_id", "speaker"],
    how="inner",
    validate="many_to_one"
)

assert len(check) == 2000
assert check.groupby("audio_id")[
    "validation_fold"
].nunique().eq(1).all()

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv_safely(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing CSV differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv_safely(speaker_stats, SPEAKER_CSV)
save_csv_safely(assignment_df, FOLD_CSV)

summary = {
    "status": "candidate_pending_review",
    "source_sha256": EXPECTED_SHA,
    "unique_audios": 400,
    "unique_speakers": 20,
    "number_of_folds": 5,
    "speaker_disjoint": True,
    "audio_hypotheses_same_fold": True,
    "official_test_split_used": False,
    "overall_mean_fluency": float(audio["fluency"].mean()),
    "overall_sd_fluency": float(audio["fluency"].std()),
    "speaker_mean_min": float(
        speaker_stats["mean_fluency"].min()
    ),
    "speaker_mean_max": float(
        speaker_stats["mean_fluency"].max()
    ),
    "fold_details": fold_rows,
    "speaker_csv_sha256": sha256(SPEAKER_CSV),
    "fold_csv_sha256": sha256(FOLD_CSV),
    "interpretation": (
        "Candidate 5-fold speaker-disjoint partition. "
        "No model fitting or outcome-based feature selection."
    )
}

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert previous == summary, (
        "Existing report differs; preserved."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_speaker_split"] = "candidate_pending_review"
progress["step5_speaker_split_sha256"] = sha256(FOLD_CSV)
progress["next_task"] = (
    "Review speaker-level fluency distribution and "
    "candidate GroupKFold assignment before model training"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.1 SPEAKER AUDIT ===")
print("Unique audios:", len(audio))
print("Unique speakers:", audio["speaker"].nunique())
print("Overall mean fluency:", round(audio["fluency"].mean(), 4))
print("Overall SD fluency:", round(audio["fluency"].std(), 4))

print("\n=== SPEAKER-LEVEL FLUENCY ===")
print(
    speaker_stats[
        [
            "speaker", "audio_count", "mean_fluency",
            "sd_fluency", "min_fluency", "max_fluency"
        ]
    ].round(3).to_string(index=False)
)

print("\n=== CANDIDATE 5-FOLD DISTRIBUTION ===")
print(
    fold_stats.round(3).to_string(index=False)
)

print("\n=== SPEAKER-DISJOINT VALIDATION ===")
print("Train/validation speaker overlap: 0")
print("Each audio assigned to one fold: PASS")
print("H1-H5 remain in same fold: PASS")
print("Official test split: UNTOUCHED")
print("Split status: CANDIDATE, NOT YET FROZEN")
print("Fold CSV:", FOLD_CSV)
print("SHA256:", sha256(FOLD_CSV))
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

DATASET = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
CANDIDATE = PROJECT / "data/splits/speaker_groupkfold5_candidate.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

FROZEN = PROJECT / "data/splits/speaker_groupkfold5_frozen.csv"
PROTOCOL = PROJECT / "configs/speaker_disjoint_cv_v1.json"

DATA_SHA = "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746"
CANDIDATE_SHA = "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7"
DESIGN_SHA = "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(DATASET) == DATA_SHA
assert sha256(CANDIDATE) == CANDIDATE_SHA
assert sha256(DESIGN) == DESIGN_SHA

modeling = pd.read_csv(
    DATASET,
    dtype={"audio_id": "string", "speaker": "string"}
)

candidate = pd.read_csv(
    CANDIDATE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(modeling) == 2000
assert len(candidate) == 400
assert candidate["audio_id"].is_unique

assert candidate["validation_fold"].between(1, 5).all()
assert set(candidate["validation_fold"]) == set(range(1, 6))

keys = ["pilot_index", "audio_id", "speaker", "fluency"]

audio = modeling[keys].drop_duplicates()

assert len(audio) == 400
assert audio["audio_id"].is_unique
assert audio["speaker"].nunique() == 20

# Verify the candidate contains exactly the same audios,
# speaker IDs and original human labels.
audit = audio.merge(
    candidate[keys + ["validation_fold"]],
    on=keys,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

assert len(audit) == 400
assert audit["_merge"].eq("both").all()

# One speaker must belong to one validation fold only.
assert candidate.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

fold_details = []

for fold in range(1, 6):

    val = candidate.loc[
        candidate["validation_fold"] == fold
    ]
    train = candidate.loc[
        candidate["validation_fold"] != fold
    ]

    val_speakers = set(val["speaker"])
    train_speakers = set(train["speaker"])

    assert len(val_speakers) == 4
    assert len(train_speakers) == 16
    assert len(val) == 80
    assert len(train) == 320
    assert train_speakers.isdisjoint(val_speakers)

    fold_details.append({
        "fold": fold,
        "train_audio_count": len(train),
        "validation_audio_count": len(val),
        "train_speaker_count": len(train_speakers),
        "validation_speaker_count": len(val_speakers),
        "validation_speakers": sorted(val_speakers),
        "train_mean_fluency": float(train["fluency"].mean()),
        "validation_mean_fluency": float(val["fluency"].mean())
    })

# Verify hypothesis-level assignment without changing it.
assigned = modeling.merge(
    candidate[["audio_id", "validation_fold"]],
    on="audio_id",
    how="inner",
    validate="many_to_one"
)

assert len(assigned) == 2000
assert assigned.groupby("audio_id")[
    "validation_fold"
].nunique().eq(1).all()
assert assigned.groupby("audio_id")[
    "hypothesis_id"
].nunique().eq(5).all()

# Freeze the exact CSV bytes; never regenerate this split.
FROZEN.parent.mkdir(parents=True, exist_ok=True)

if FROZEN.exists():
    assert sha256(FROZEN) == CANDIDATE_SHA, (
        "Frozen split differs. Original file preserved."
    )
else:
    tmp = FROZEN.with_suffix(".tmp")
    tmp.write_bytes(CANDIDATE.read_bytes())
    assert sha256(tmp) == CANDIDATE_SHA
    tmp.replace(FROZEN)

protocol = {
    "name": "speaker_disjoint_cv_v1",
    "status": "frozen",
    "dataset": "mispeech/speechocean762",
    "selected_audio_count": 400,
    "unique_speakers": 20,
    "number_of_folds": 5,
    "validation_speakers_per_fold": 4,
    "validation_audios_per_fold": 80,
    "group_key": "speaker",
    "split_method": "GroupKFold",
    "split_source": "precomputed_candidate_csv",
    "sklearn_version_at_freeze": sklearn.__version__,
    "random_seed_required_for_reusing_saved_split": False,
    "speaker_disjoint": True,
    "same_audio_hypotheses_same_fold": True,
    "official_test_split_untouched": True,
    "modeling_dataset_sha256": DATA_SHA,
    "frozen_split_sha256": CANDIDATE_SHA,
    "feature_design_sha256": DESIGN_SHA,
    "primary_models": ["A", "B1", "C1"],
    "secondary_ablation_models": ["B2", "C2"],
    "training_rules": {
        "all_validation_speakers_unseen_in_training": True,
        "fit_imputation_on_training_only": True,
        "fit_scaling_on_training_only": True,
        "tune_hyperparameters_with_training_speakers_only": True,
        "model_A_one_row_per_audio": True,
        "model_B_C_repeated_hypotheses_grouped_by_audio": True,
        "evaluate_each_hypothesis_separately": True,
        "no_validation_label_based_feature_selection": True
    },
    "evaluation_plan": {
        "primary_metrics": ["MAE", "RMSE"],
        "additional_metrics": [
            "speaker_level_MAE",
            "per_fold_MAE",
            "prediction_range_across_H1_H5",
            "prediction_sd_across_H1_H5"
        ],
        "acoustic_model_expected_asr_invariance": True
    },
    "fold_details": fold_details
}

PROTOCOL.parent.mkdir(parents=True, exist_ok=True)

if PROTOCOL.exists():
    previous = json.loads(
        PROTOCOL.read_text(encoding="utf-8")
    )
    assert previous == protocol, (
        "Existing frozen CV protocol differs; preserved."
    )
else:
    tmp = PROTOCOL.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(protocol, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(PROTOCOL)

progress = json.loads(CHECK.read_text(encoding="utf-8"))

progress["step5_speaker_split"] = "frozen"
progress["step5_speaker_split_sha256"] = sha256(FROZEN)
progress["step5_cv_protocol_sha256"] = sha256(PROTOCOL)
progress["next_task"] = (
    "Define leakage-safe training and evaluation pipelines "
    "for models A, B1, B2, C1 and C2"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.2 SPEAKER CV FROZEN ===")
print("Status:", protocol["status"])
print("Audios:", protocol["selected_audio_count"])
print("Speakers:", protocol["unique_speakers"])
print("Folds:", protocol["number_of_folds"])
print("Validation speakers per fold:", 4)
print("Validation audios per fold:", 80)
print("Train/validation speaker overlap: ZERO")
print("H1-H5 partition integrity: PASS")
print("Official test split: UNTOUCHED")

print("\n=== FROZEN FOLD SUMMARY ===")
for item in fold_details:
    print(
        f"Fold {item['fold']}: "
        f"val_speakers={item['validation_speakers']} "
        f"val_mean={item['validation_mean_fluency']:.3f}"
    )

print("\nFrozen split:", FROZEN)
print("Split SHA256:", sha256(FROZEN))
print("Protocol:", PROTOCOL)
print("Protocol SHA256:", sha256(PROTOCOL))
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

DATA = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = PROJECT / "data/splits/speaker_groupkfold5_frozen.csv"
CV_CONFIG = PROJECT / "configs/speaker_disjoint_cv_v1.json"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

RESULTS = PROJECT / "results/step05"
PREDICTIONS = RESULTS / "baseline_acoustic_oof_predictions.csv"
FOLD_RESULTS = RESULTS / "baseline_acoustic_fold_metrics.csv"
SPEAKER_RESULTS = RESULTS / "baseline_acoustic_speaker_metrics.csv"
REPORT = PROJECT / "reports/step05_baseline_acoustic_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    CV_CONFIG: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected_hash in EXPECTED.items():
    assert sha256(path) == expected_hash, (
        f"Source SHA256 mismatch: {path}"
    )

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv_config = json.loads(CV_CONFIG.read_text(encoding="utf-8"))

assert cv_config["status"] == "frozen"
assert cv_config["number_of_folds"] == 5
assert cv_config["speaker_disjoint"] is True

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)

split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

ids = ["pilot_index", "audio_id", "speaker", "fluency"]

audio = data[ids].drop_duplicates()

assert len(audio) == 400
assert audio["audio_id"].is_unique

acoustic_features = [
    col for col in data.columns
    if col not in (
        ids
        + ["hypothesis_id"]
        + design["linguistic_feature_names"]
        + design["prompt_feature_names"]
    )
]

assert len(acoustic_features) == 14
assert len(set(acoustic_features)) == 14
assert "fluency" not in acoustic_features

# Reduce to one acoustic record per audio.
acoustic = data.loc[
    data["hypothesis_id"] == "H1",
    ids + acoustic_features
].copy()

assert len(acoustic) == 400

df = acoustic.merge(
    split[ids + ["validation_fold"]],
    on=ids,
    how="inner",
    validate="one_to_one"
)

assert len(df) == 400
assert df["validation_fold"].between(1, 5).all()

prediction_rows = []
fold_rows = []

# Fixed alpha for the initial benchmark.
RIDGE_ALPHA = 1.0

for fold in range(1, 6):

    train = df.loc[
        df["validation_fold"] != fold
    ].copy()

    val = df.loc[
        df["validation_fold"] == fold
    ].copy()

    assert len(train) == 320
    assert len(val) == 80

    assert set(train["speaker"]).isdisjoint(
        set(val["speaker"])
    )

    X_train = train[acoustic_features]
    X_val = val[acoustic_features]

    y_train = train["fluency"].to_numpy(dtype=float)
    y_val = val["fluency"].to_numpy(dtype=float)

    # M0: training-fold mean only.
    pred_mean = np.full(
        len(val),
        y_train.mean(),
        dtype=float
    )

    # A: imputation and scaling learned on training only.
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=RIDGE_ALPHA))
    ])

    model.fit(X_train, y_train)
    pred_acoustic = model.predict(X_val)

    assert np.isfinite(pred_acoustic).all()

    fold_rows.append({
        "fold": fold,
        "train_audio_count": len(train),
        "validation_audio_count": len(val),
        "train_speaker_count": train["speaker"].nunique(),
        "validation_speaker_count": val["speaker"].nunique(),
        "train_mean_fluency": float(y_train.mean()),
        "validation_mean_fluency": float(y_val.mean()),
        "mean_baseline_mae": float(
            mean_absolute_error(y_val, pred_mean)
        ),
        "mean_baseline_rmse": float(
            np.sqrt(mean_squared_error(y_val, pred_mean))
        ),
        "acoustic_ridge_mae": float(
            mean_absolute_error(y_val, pred_acoustic)
        ),
        "acoustic_ridge_rmse": float(
            np.sqrt(mean_squared_error(y_val, pred_acoustic))
        )
    })

    for i, (_, row) in enumerate(val.iterrows()):
        prediction_rows.append({
            "pilot_index": int(row["pilot_index"]),
            "audio_id": str(row["audio_id"]),
            "speaker": str(row["speaker"]),
            "validation_fold": fold,
            "fluency": float(y_val[i]),
            "mean_baseline_prediction": float(pred_mean[i]),
            "acoustic_ridge_prediction": float(pred_acoustic[i])
        })

predictions = pd.DataFrame(prediction_rows).sort_values(
    "pilot_index"
).reset_index(drop=True)

fold_metrics = pd.DataFrame(fold_rows)

assert len(predictions) == 400
assert predictions["audio_id"].is_unique
assert predictions["speaker"].nunique() == 20
assert predictions["validation_fold"].nunique() == 5
assert predictions["fluency"].notna().all()

# Each speaker has predictions from exactly one fold.
assert predictions.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

y = predictions["fluency"].to_numpy()

overall_metrics = {}

for name, col in [
    ("M0_mean_baseline", "mean_baseline_prediction"),
    ("A_acoustic_ridge", "acoustic_ridge_prediction")
]:
    pred = predictions[col].to_numpy()

    overall_metrics[name] = {
        "MAE": float(mean_absolute_error(y, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred)))
    }

speaker_rows = []

for speaker, group in predictions.groupby("speaker"):
    speaker_rows.append({
        "speaker": str(speaker),
        "fold": int(group["validation_fold"].iloc[0]),
        "audio_count": len(group),
        "mean_fluency": float(group["fluency"].mean()),
        "mean_baseline_mae": float(
            mean_absolute_error(
                group["fluency"],
                group["mean_baseline_prediction"]
            )
        ),
        "acoustic_ridge_mae": float(
            mean_absolute_error(
                group["fluency"],
                group["acoustic_ridge_prediction"]
            )
        )
    })

speaker_metrics = pd.DataFrame(speaker_rows)

assert len(speaker_metrics) == 20

RESULTS.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv_safely(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")

    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing result differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv_safely(predictions, PREDICTIONS)
save_csv_safely(fold_metrics, FOLD_RESULTS)
save_csv_safely(speaker_metrics, SPEAKER_RESULTS)

report = {
    "status": "PASS",
    "stage": "initial_fixed_alpha_benchmark",
    "model_A": "Ridge",
    "ridge_alpha": RIDGE_ALPHA,
    "hyperparameter_tuning_performed": False,
    "validation": "frozen_5fold_speaker_disjoint",
    "unique_audio_count": 400,
    "unique_speaker_count": 20,
    "acoustic_feature_count": 14,
    "prediction_count_per_model": 400,
    "metrics": overall_metrics,
    "predictions_sha256": sha256(PREDICTIONS),
    "fold_metrics_sha256": sha256(FOLD_RESULTS),
    "speaker_metrics_sha256": sha256(SPEAKER_RESULTS),
    "modeling_dataset_sha256": sha256(DATA),
    "frozen_split_sha256": sha256(SPLIT),
    "note": (
        "Predictions are out-of-fold for unseen speakers. "
        "Ridge alpha is fixed at 1.0, without tuning. "
        "No official test data were used."
    )
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, (
        "Existing benchmark report differs."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_initial_acoustic_benchmark"] = "qc_pass"
progress["step5_initial_acoustic_predictions_sha256"] = sha256(PREDICTIONS)
progress["next_task"] = (
    "Review acoustic Ridge and mean baseline results "
    "before transcript-only Ridge experiments"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.3 INITIAL ACOUSTIC BENCHMARK ===")
print("Status:", report["status"])
print("Validation: frozen 5-fold speaker-disjoint")
print("Unique audios:", 400)
print("Features:", len(acoustic_features))
print("Ridge alpha:", RIDGE_ALPHA)
print("Hyperparameter tuning: NO")

print("\n=== OVERALL OUT-OF-FOLD METRICS ===")
for name, metrics in overall_metrics.items():
    print(
        f"{name}: "
        f"MAE={metrics['MAE']:.4f}, "
        f"RMSE={metrics['RMSE']:.4f}"
    )

print("\n=== PER-FOLD METRICS ===")
print(fold_metrics.round(4).to_string(index=False))

print("\n=== PER-SPEAKER MAE ===")
print(speaker_metrics.round(4).to_string(index=False))

print("\nPredictions CSV:", PREDICTIONS)
print("Predictions SHA256:", sha256(PREDICTIONS))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
DATA = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = PROJECT / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CV = PROJECT / "configs/speaker_disjoint_cv_v1.json"
ACOUSTIC_PRED = PROJECT / "results/step05/baseline_acoustic_oof_predictions.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

RESULT_DIR = PROJECT / "results/step05"
PRED_OUT = RESULT_DIR / "b1_transcript_ridge_oof_predictions.csv"
FOLD_OUT = RESULT_DIR / "b1_transcript_ridge_fold_metrics.csv"
SPEAKER_OUT = RESULT_DIR / "b1_transcript_ridge_speaker_metrics.csv"
REPORT = PROJECT / "reports/step05_b1_transcript_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    ACOUSTIC_PRED: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

assert cv["status"] == "frozen"
assert cv["speaker_disjoint"] is True

FEATURES = design["linguistic_feature_names"]
assert len(FEATURES) == 7

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(data) == 2000
assert len(split) == 400
assert all(c in data.columns for c in FEATURES)

keys = ["pilot_index", "audio_id", "speaker", "fluency"]

df = data.merge(
    split[keys + ["validation_fold"]],
    on=keys,
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()

predictions = []
fold_metrics = []

ALPHA = 1.0

for fold in range(1, 6):

    train = df.loc[df["validation_fold"] != fold].copy()
    val = df.loc[df["validation_fold"] == fold].copy()

    assert len(train) == 1600
    assert len(val) == 400
    assert train["audio_id"].nunique() == 320
    assert val["audio_id"].nunique() == 80
    assert train["speaker"].nunique() == 16
    assert val["speaker"].nunique() == 4

    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    pred = model.predict(val[FEATURES])
    assert np.isfinite(pred).all()

    result = val[
        keys + ["hypothesis_id", "validation_fold"]
    ].copy()
    result["b1_prediction"] = pred
    predictions.append(result)

    # Each fold has 80 audios and exactly five
    # predictions per audio.
    fold_metrics.append({
        "fold": fold,
        "train_speakers": 16,
        "validation_speakers": 4,
        "train_unique_audios": 320,
        "validation_unique_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(mean_absolute_error(val["fluency"], pred)),
        "RMSE": float(np.sqrt(
            mean_squared_error(val["fluency"], pred)
        ))
    })

oof = pd.concat(predictions, ignore_index=True).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

fold_table = pd.DataFrame(fold_metrics)

assert len(oof) == 2000
assert not oof.duplicated(["audio_id", "hypothesis_id"]).any()
assert oof["audio_id"].nunique() == 400

# Absolute errors for every hypothesis.
oof["absolute_error"] = (
    oof["fluency"] - oof["b1_prediction"]
).abs()

# Per-audio instability of B1 prediction across H1-H5.
audio_stability = oof.groupby("audio_id")["b1_prediction"].agg(
    ["min", "max", "std"]
)

audio_stability["prediction_range"] = (
    audio_stability["max"] - audio_stability["min"]
)

# Per-hypothesis accuracy on the same 400 audios.
hypothesis_metrics = {}
for hid, group in oof.groupby("hypothesis_id"):
    hypothesis_metrics[hid] = {
        "MAE": float(mean_absolute_error(
            group["fluency"], group["b1_prediction"]
        )),
        "RMSE": float(np.sqrt(mean_squared_error(
            group["fluency"], group["b1_prediction"]
        )))
    }

# Speaker-level MAE: 20 independent speaker groups.
speaker_rows = []
for speaker, group in oof.groupby("speaker"):
    speaker_rows.append({
        "speaker": str(speaker),
        "fold": int(group["validation_fold"].iloc[0]),
        "unique_audios": group["audio_id"].nunique(),
        "hypothesis_rows": len(group),
        "MAE": float(group["absolute_error"].mean())
    })

speaker_table = pd.DataFrame(speaker_rows)
assert len(speaker_table) == 20

overall = {
    "MAE": float(mean_absolute_error(
        oof["fluency"], oof["b1_prediction"]
    )),
    "RMSE": float(np.sqrt(mean_squared_error(
        oof["fluency"], oof["b1_prediction"]
    ))),
    "mean_prediction_range": float(
        audio_stability["prediction_range"].mean()
    ),
    "median_prediction_range": float(
        audio_stability["prediction_range"].median()
    ),
    "mean_prediction_sd": float(
        audio_stability["std"].mean()
    ),
    "audios_with_nonzero_prediction_range": int(
        (audio_stability["prediction_range"] > 1e-10).sum()
    )
}

# Compare with previously saved acoustic-only OOF predictions.
a = pd.read_csv(
    ACOUSTIC_PRED,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(a) == 400
assert a["audio_id"].is_unique

comparison = oof.merge(
    a[["audio_id", "acoustic_ridge_prediction"]],
    on="audio_id",
    how="inner",
    validate="many_to_one"
)

assert len(comparison) == 2000
assert comparison["fluency"].notna().all()

a_mae = float(mean_absolute_error(
    comparison["fluency"],
    comparison["acoustic_ridge_prediction"]
))

# The acoustic value is repeated five times for
# comparison, not five independent observations.
assert np.isclose(a_mae, 0.9399, atol=0.0001)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing result differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv(oof, PRED_OUT)
save_csv(fold_table, FOLD_OUT)
save_csv(speaker_table, SPEAKER_OUT)

report = {
    "status": "PASS",
    "model": "B1_transcript_only_ridge",
    "alpha": ALPHA,
    "hyperparameter_tuning": False,
    "feature_names": FEATURES,
    "feature_count": 7,
    "speaker_disjoint_folds": 5,
    "train_preprocessing_only": True,
    "training_hypotheses_per_audio": 5,
    "unique_validation_audios": 400,
    "validation_hypothesis_rows": 2000,
    "overall_metrics": overall,
    "per_hypothesis_metrics": hypothesis_metrics,
    "reference_acoustic_MAE": a_mae,
    "source_modeling_sha256": sha256(DATA),
    "source_frozen_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED_OUT),
    "fold_metrics_sha256": sha256(FOLD_OUT),
    "speaker_metrics_sha256": sha256(SPEAKER_OUT),
    "note": (
        "All five hypotheses used as repeated training examples. "
        "Each audio has equal total weight because each has "
        "five hypotheses. Validation speakers remain unseen. "
        "Hypothesis rows are not independent observations."
    )
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, "Existing report differs"
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_b1_transcript_ridge"] = "qc_pass"
progress["step5_b1_predictions_sha256"] = sha256(PRED_OUT)
progress["next_task"] = (
    "Review B1 accuracy and prediction instability; "
    "then evaluate prompt-aware B2 and fusion C1/C2"
)
progress["updated_at"] = (
    __import__("datetime").datetime.now(
        __import__("datetime").timezone.utc
    ).isoformat()
)

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.4 B1 TRANSCRIPT RIDGE SUMMARY ===")
print("Status:", report["status"])
print("Unique audios:", overall and 400)
print("Speakers:", 20)
print("OOF hypothesis predictions:", len(oof))
print("Features:", len(FEATURES))
print("Alpha:", ALPHA)
print("Speaker-disjoint validation: PASS")

print("\n=== OVERALL COMPARISON ===")
print(f"A acoustic MAE: {a_mae:.4f}")
print(f"B1 transcript MAE: {overall['MAE']:.4f}")
print(f"B1 transcript RMSE: {overall['RMSE']:.4f}")

print("\n=== H1-H5 INDIVIDUAL METRICS ===")
for hid, metrics in sorted(hypothesis_metrics.items()):
    print(
        f"{hid}: MAE={metrics['MAE']:.4f}, "
        f"RMSE={metrics['RMSE']:.4f}"
    )

print("\n=== PREDICTION INSTABILITY ===")
for key in [
    "mean_prediction_range",
    "median_prediction_range",
    "mean_prediction_sd",
    "audios_with_nonzero_prediction_range"
]:
    print(f"{key}: {overall[key]}")

print("\n=== FOLD METRICS ===")
print(fold_table.round(4).to_string(index=False))

print("\n=== SPEAKER METRICS ===")
print(speaker_table.round(4).to_string(index=False))

print("\nPredictions:", PRED_OUT)
print("SHA256:", sha256(PRED_OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "results/step05/b1_transcript_ridge_oof_predictions.csv"
MODEL_REPORT = PROJECT / "reports/step05_b1_transcript_ridge_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "results/step05"
AUDIO_OUT = OUT_DIR / "b1_prediction_instability_per_audio.csv"
REPORT = PROJECT / "reports/step05_b1_outlier_influence_audit.json"

EXPECTED_SHA = (
    "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_SHA

previous = json.loads(MODEL_REPORT.read_text(encoding="utf-8"))
assert previous["status"] == "PASS"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert not df.duplicated(["audio_id", "hypothesis_id"]).any()

required = ["fluency", "b1_prediction", "validation_fold"]
assert df[required].notna().all().all()

df["signed_error"] = df["b1_prediction"] - df["fluency"]
df["absolute_error_check"] = df["signed_error"].abs()
df["squared_error"] = df["signed_error"] ** 2

def metrics(frame):
    return {
        "n": int(len(frame)),
        "MAE": float(frame["absolute_error_check"].mean()),
        "RMSE": float(np.sqrt(frame["squared_error"].mean()))
    }

all_metrics = metrics(df)

assert np.isclose(
    all_metrics["MAE"],
    previous["overall_metrics"]["MAE"],
    atol=1e-10
)
assert np.isclose(
    all_metrics["RMSE"],
    previous["overall_metrics"]["RMSE"],
    atol=1e-10
)

critical_mask = (
    df["audio_id"].eq("030210164") &
    df["hypothesis_id"].eq("H2")
)

assert critical_mask.sum() == 1

critical = df.loc[critical_mask].iloc[0]
without_one = df.loc[~critical_mask].copy()

# Diagnostic only: 1999 predictions, unequal
# hypothesis counts for the affected audio.
without_one_metrics = metrics(without_one)

# Audio-level sensitivity: remove all five
# hypotheses for the affected audio.
without_audio = df.loc[
    ~df["audio_id"].eq("030210164")
].copy()
without_audio_metrics = metrics(without_audio)

total_sse = float(df["squared_error"].sum())
critical_sse = float(critical["squared_error"])

critical_sse_share = critical_sse / total_sse

# Complete, equal-weight 400-audio stability table.
audio = (
    df.groupby(
        ["pilot_index", "audio_id", "speaker", "validation_fold"],
        as_index=False
    )
    .agg(
        human_fluency=("fluency", "first"),
        prediction_min=("b1_prediction", "min"),
        prediction_max=("b1_prediction", "max"),
        prediction_mean=("b1_prediction", "mean"),
        prediction_sd=("b1_prediction", "std"),
        mean_absolute_error=("absolute_error_check", "mean"),
        max_absolute_error=("absolute_error_check", "max"),
        hypothesis_count=("hypothesis_id", "count")
    )
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

audio = audio.sort_values(
    ["pilot_index"]
).reset_index(drop=True)

top_errors = df.sort_values(
    "squared_error", ascending=False
).head(10)

top_ranges = audio.sort_values(
    "prediction_range", ascending=False
).head(10)

# Separate analysis for the beam-search hypothesis.
h2 = df.loc[df["hypothesis_id"] == "H2"]
h2_without = h2.loc[
    ~h2["audio_id"].eq("030210164")
]

# Fold 2 includes this critical audio.
critical_fold = int(critical["validation_fold"])
fold_df = df.loc[
    df["validation_fold"] == critical_fold
]

fold_without_one = fold_df.loc[
    ~critical_mask.loc[fold_df.index]
]

OUT_DIR.mkdir(parents=True, exist_ok=True)

tmp = AUDIO_OUT.with_suffix(".tmp")
audio.to_csv(tmp, index=False, encoding="utf-8")

if AUDIO_OUT.exists():
    assert sha256(AUDIO_OUT) == sha256(tmp), (
        "Existing audio instability table differs."
    )
    tmp.unlink()
else:
    tmp.replace(AUDIO_OUT)

report = {
    "status": "PASS",
    "model": "B1_transcript_only_ridge",
    "original_predictions_unchanged": True,
    "full_metrics": all_metrics,
    "without_one_h2_diagnostic": without_one_metrics,
    "without_entire_audio_diagnostic": without_audio_metrics,
    "critical_audio_id": "030210164",
    "critical_hypothesis_id": "H2",
    "critical_speaker": str(critical["speaker"]),
    "critical_fold": critical_fold,
    "critical_human_fluency": float(critical["fluency"]),
    "critical_prediction": float(critical["b1_prediction"]),
    "critical_absolute_error": float(critical["absolute_error_check"]),
    "critical_squared_error": critical_sse,
    "critical_sse_share": critical_sse_share,
    "h2_full": metrics(h2),
    "h2_without_critical": metrics(h2_without),
    "critical_fold_full": metrics(fold_df),
    "critical_fold_without_one": metrics(fold_without_one),
    "mean_prediction_range": float(audio["prediction_range"].mean()),
    "median_prediction_range": float(audio["prediction_range"].median()),
    "p95_prediction_range": float(
        audio["prediction_range"].quantile(0.95)
    ),
    "max_prediction_range": float(audio["prediction_range"].max()),
    "variable_prediction_audios": int(
        (audio["prediction_range"] > 1e-10).sum()
    ),
    "source_sha256": sha256(SOURCE),
    "audio_table_sha256": sha256(AUDIO_OUT),
    "interpretation": (
        "Post-hoc influence diagnostics only. "
        "No observations excluded from primary model evaluation. "
        "The same fitted OOF models are used."
    )
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    saved = json.loads(REPORT.read_text(encoding="utf-8"))
    assert saved == report, "Existing report differs."
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_b1_outlier_audit"] = "qc_pass"
progress["step5_b1_outlier_report_sha256"] = sha256(REPORT)
progress["next_task"] = (
    "Review B1 outlier influence and proceed to B2/C1/C2 "
    "with the unchanged frozen speaker-disjoint folds"
)

from datetime import datetime, timezone
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.5 B1 OUTLIER INFLUENCE AUDIT ===")
print("Status: PASS")
print("Critical audio:", report["critical_audio_id"])
print("Critical hypothesis:", report["critical_hypothesis_id"])
print("Speaker:", report["critical_speaker"])
print("Fold:", report["critical_fold"])
print("Human fluency:", report["critical_human_fluency"])
print("B1 prediction:", round(report["critical_prediction"], 4))
print("Absolute error:", round(report["critical_absolute_error"], 4))
print("Squared error share (%):", round(100 * critical_sse_share, 2))

print("\n=== GLOBAL DIAGNOSTIC COMPARISON ===")
for name, values in [
    ("All 2000 predictions", all_metrics),
    ("Without one H2 prediction", without_one_metrics),
    ("Without entire critical audio", without_audio_metrics)
]:
    print(
        f"{name}: n={values['n']}, "
        f"MAE={values['MAE']:.4f}, RMSE={values['RMSE']:.4f}"
    )

print("\n=== H2 AND CRITICAL FOLD ===")
for name, values in [
    ("H2 full", metrics(h2)),
    ("H2 without one", metrics(h2_without)),
    ("Critical fold full", metrics(fold_df)),
    ("Critical fold without one", metrics(fold_without_one))
]:
    print(
        f"{name}: n={values['n']}, "
        f"MAE={values['MAE']:.4f}, RMSE={values['RMSE']:.4f}"
    )

print("\n=== TOP 10 INDIVIDUAL ERRORS ===")
print(
    top_errors[
        ["audio_id", "speaker", "hypothesis_id",
         "fluency", "b1_prediction",
         "absolute_error_check", "squared_error"]
    ].round(4).to_string(index=False)
)

print("\n=== TOP 10 PREDICTION RANGES ===")
print(
    top_ranges[
        ["audio_id", "speaker", "prediction_range",
         "prediction_sd", "mean_absolute_error"]
    ].round(4).to_string(index=False)
)

print("\n=== STABILITY SUMMARY ===")
print(
    "Mean range:",
    round(report["mean_prediction_range"], 4)
)
print(
    "Median range:",
    round(report["median_prediction_range"], 4)
)
print(
    "95th percentile range:",
    round(report["p95_prediction_range"], 4)
)
print(
    "Maximum range:",
    round(report["max_prediction_range"], 4)
)
print(
    "Variable prediction audios:",
    report["variable_prediction_audios"]
)

print("\nSaved audio table:", AUDIO_OUT)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"
A_PRED = P / "results/step05/baseline_acoustic_oof_predictions.csv"
B1_PRED = P / "results/step05/b1_transcript_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step05"
PRED = OUTDIR / "c1_fusion_ridge_oof_predictions.csv"
FOLDS = OUTDIR / "c1_fusion_ridge_fold_metrics.csv"
AUDIO = OUTDIR / "c1_fusion_prediction_stability_per_audio.csv"
REPORT = P / "reports/step05_c1_fusion_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    A_PRED: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    B1_PRED: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
protocol = json.loads(CV.read_text(encoding="utf-8"))

assert protocol["status"] == "frozen"
assert design["models"]["C1"]["feature_groups"] == [
    "acoustic", "linguistic"
]

data = pd.read_csv(
    DATA, dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT, dtype={"audio_id": "string", "speaker": "string"}
)

ID = ["pilot_index", "audio_id", "speaker"]
KEYS = ID + ["hypothesis_id"]
LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURES = ACOUSTIC + LING

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(FEATURES) == 21
assert "fluency" not in FEATURES
assert not set(FEATURES).intersection(KEYS + PROMPT)

assert len(data) == 2000
assert len(split) == 400
assert not data.duplicated(KEYS).any()

df = data.merge(
    split[ID + ["fluency", "validation_fold"]],
    on=ID + ["fluency"],
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()

ALPHA = 1.0
pred_parts = []
fold_rows = []

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold]
    val = df.loc[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert train["audio_id"].nunique() == 320
    assert val["audio_id"].nunique() == 80

    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    predictions = model.predict(val[FEATURES])
    assert np.isfinite(predictions).all()

    part = val[KEYS + ["fluency", "validation_fold"]].copy()
    part["c1_prediction"] = predictions
    pred_parts.append(part)

    fold_rows.append({
        "fold": fold,
        "validation_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(mean_absolute_error(
            val["fluency"], predictions
        )),
        "RMSE": float(np.sqrt(mean_squared_error(
            val["fluency"], predictions
        )))
    })

oof = pd.concat(pred_parts, ignore_index=True).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(oof) == 2000
assert oof["audio_id"].nunique() == 400
assert not oof.duplicated(["audio_id", "hypothesis_id"]).any()

# Verify C1 rows against earlier acoustic and transcript OOF.
a = pd.read_csv(
    A_PRED, dtype={"audio_id": "string", "speaker": "string"}
)
b = pd.read_csv(
    B1_PRED, dtype={"audio_id": "string", "speaker": "string"}
)

comp = oof.merge(
    a[ID + ["fluency", "validation_fold", "acoustic_ridge_prediction"]],
    on=ID + ["fluency", "validation_fold"],
    validate="many_to_one"
).merge(
    b[KEYS + ["fluency", "validation_fold", "b1_prediction"]],
    on=KEYS + ["fluency", "validation_fold"],
    validate="one_to_one"
)

assert len(comp) == 2000

def metrics(y, pred):
    return {
        "MAE": float(mean_absolute_error(y, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred)))
    }

comparison = {
    "A_acoustic": metrics(
        comp["fluency"], comp["acoustic_ridge_prediction"]
    ),
    "B1_transcript": metrics(
        comp["fluency"], comp["b1_prediction"]
    ),
    "C1_fusion": metrics(
        comp["fluency"], comp["c1_prediction"]
    )
}

audio = (
    oof.groupby(ID + ["validation_fold"], as_index=False)
    .agg(
        fluency=("fluency", "first"),
        prediction_min=("c1_prediction", "min"),
        prediction_max=("c1_prediction", "max"),
        prediction_sd=("c1_prediction", "std"),
        hypothesis_count=("hypothesis_id", "count")
    )
)

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

hyp_metrics = {
    h: metrics(
        g["fluency"], g["c1_prediction"]
    )
    for h, g in oof.groupby("hypothesis_id")
}

stability = {
    "mean_range": float(audio["prediction_range"].mean()),
    "median_range": float(audio["prediction_range"].median()),
    "p95_range": float(audio["prediction_range"].quantile(0.95)),
    "max_range": float(audio["prediction_range"].max()),
    "variable_audio_count": int(
        (audio["prediction_range"] > 1e-10).sum()
    )
}

critical = oof.loc[
    oof["audio_id"].eq("030210164") &
    oof["hypothesis_id"].eq("H2")
]
assert len(critical) == 1

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(oof, PRED)
save_csv(pd.DataFrame(fold_rows), FOLDS)
save_csv(audio, AUDIO)

report = {
    "status": "PASS",
    "model": "C1_acoustic_linguistic_fusion_ridge",
    "alpha": ALPHA,
    "hyperparameter_tuning": False,
    "feature_count": 21,
    "acoustic_feature_count": 14,
    "linguistic_feature_count": 7,
    "validation": "frozen_5fold_speaker_disjoint",
    "OOF_prediction_rows": 2000,
    "unique_audios": 400,
    "comparison": comparison,
    "per_hypothesis_metrics": hyp_metrics,
    "stability": stability,
    "critical_audio_H2_prediction": float(
        critical["c1_prediction"].iloc[0]
    ),
    "source_dataset_sha256": sha256(DATA),
    "source_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED),
    "fold_metrics_sha256": sha256(FOLDS),
    "audio_stability_sha256": sha256(AUDIO),
    "note": (
        "No clipping or hyperparameter tuning. "
        "Training-only imputation and scaling. "
        "All five ASR hypotheses remain in the same speaker fold."
    )
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, (
        "Existing report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_c1_fusion_ridge"] = "qc_pass"
progress["step5_c1_predictions_sha256"] = sha256(PRED)
progress["next_task"] = (
    "Compare C1 fusion accuracy and ASR-induced instability "
    "against models A and B1"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.6 C1 FUSION RIDGE SUMMARY ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print("Unique audios: 400")
print("OOF predictions:", len(oof))
print("Speaker-disjoint CV: PASS")
print("Ridge alpha:", ALPHA)

print("\n=== MODEL COMPARISON ===")
for name, m in comparison.items():
    print(f"{name}: MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}")

print("\n=== C1 H1-H5 METRICS ===")
for h, m in sorted(hyp_metrics.items()):
    print(f"{h}: MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}")

print("\n=== C1 PREDICTION STABILITY ===")
for key, value in stability.items():
    print(f"{key}: {value}")

print("\n=== C1 CRITICAL ASR CASE ===")
print("Audio: 030210164")
print("Hypothesis: H2")
print("Human fluency:", float(critical["fluency"].iloc[0]))
print("C1 prediction:", float(critical["c1_prediction"].iloc[0]))

print("\n=== C1 PER-FOLD METRICS ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

print("\nPrediction CSV:", PRED)
print("SHA256:", sha256(PRED))
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
A_FILE = P / "results/step05/baseline_acoustic_oof_predictions.csv"
B_FILE = P / "results/step05/b1_transcript_ridge_oof_predictions.csv"
C_FILE = P / "results/step05/c1_fusion_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUT = P / "results/step05/c1_vs_a_paired_audio_audit.csv"
REPORT = P / "reports/step05_c1_outlier_paired_audit.json"

EXPECTED = {
    A_FILE: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    B_FILE: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    C_FILE: "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert sha256(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

a, b, c = read(A_FILE), read(B_FILE), read(C_FILE)

assert len(a) == 400
assert len(b) == len(c) == 2000

KEYS = [
    "pilot_index", "audio_id", "speaker",
    "fluency", "validation_fold"
]
HKEYS = KEYS + ["hypothesis_id"]

assert not a.duplicated(KEYS).any()
assert not b.duplicated(HKEYS).any()
assert not c.duplicated(HKEYS).any()

merged = c[HKEYS + ["c1_prediction"]].merge(
    b[HKEYS + ["b1_prediction"]],
    on=HKEYS,
    validate="one_to_one"
).merge(
    a[KEYS + ["acoustic_ridge_prediction"]],
    on=KEYS,
    validate="many_to_one"
)

assert len(merged) == 2000
assert merged["audio_id"].nunique() == 400
assert merged.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

for name, col in [
    ("A", "acoustic_ridge_prediction"),
    ("B1", "b1_prediction"),
    ("C1", "c1_prediction")
]:
    merged[f"{name}_error"] = merged[col] - merged["fluency"]
    merged[f"{name}_ae"] = merged[f"{name}_error"].abs()
    merged[f"{name}_se"] = merged[f"{name}_error"] ** 2

def metrics(frame, model):
    return {
        "n": int(len(frame)),
        "MAE": float(frame[f"{model}_ae"].mean()),
        "RMSE": float(np.sqrt(frame[f"{model}_se"].mean()))
    }

all_metrics = {
    name: metrics(merged, name)
    for name in ["A", "B1", "C1"]
}

CRITICAL_ID = "030210164"
critical_mask = (
    merged["audio_id"].eq(CRITICAL_ID) &
    merged["hypothesis_id"].eq("H2")
)

assert critical_mask.sum() == 1

critical = merged.loc[critical_mask].iloc[0]
without_one = merged.loc[~critical_mask]
without_audio = merged.loc[
    ~merged["audio_id"].eq(CRITICAL_ID)
]

diagnostics = {
    "all_predictions": all_metrics,
    "without_one_H2": {
        name: metrics(without_one, name)
        for name in ["A", "B1", "C1"]
    },
    "without_entire_audio": {
        name: metrics(without_audio, name)
        for name in ["A", "B1", "C1"]
    }
}

sse_share = {
    name: float(
        critical[f"{name}_se"] /
        merged[f"{name}_se"].sum()
    )
    for name in ["B1", "C1"]
}

# Paired audio-level error: each of 400 audios
# contributes once with five equally weighted hypotheses.
audio = merged.groupby(
    KEYS, as_index=False
).agg(
    a_mae=("A_ae", "mean"),
    b1_mae=("B1_ae", "mean"),
    c1_mae=("C1_ae", "mean"),
    a_mse=("A_se", "mean"),
    b1_mse=("B1_se", "mean"),
    c1_mse=("C1_se", "mean"),
    b1_min=("b1_prediction", "min"),
    b1_max=("b1_prediction", "max"),
    c1_min=("c1_prediction", "min"),
    c1_max=("c1_prediction", "max")
)

assert len(audio) == 400
assert audio["audio_id"].is_unique

audio["c1_minus_a_absolute_error"] = (
    audio["c1_mae"] - audio["a_mae"]
)
audio["b1_range"] = audio["b1_max"] - audio["b1_min"]
audio["c1_range"] = audio["c1_max"] - audio["c1_min"]

win_counts = {
    "C1_better_than_A": int(
        (audio["c1_minus_a_absolute_error"] < -1e-10).sum()
    ),
    "C1_worse_than_A": int(
        (audio["c1_minus_a_absolute_error"] > 1e-10).sum()
    ),
    "ties": int(
        (audio["c1_minus_a_absolute_error"].abs() <= 1e-10).sum()
    )
}
assert sum(win_counts.values()) == 400

stability = {}

for name in ["b1", "c1"]:
    ranges = audio[f"{name}_range"]
    without_critical = audio.loc[
        ~audio["audio_id"].eq(CRITICAL_ID),
        f"{name}_range"
    ]
    stability[name.upper()] = {
        "mean_range_all": float(ranges.mean()),
        "median_range_all": float(ranges.median()),
        "p95_range_all": float(ranges.quantile(0.95)),
        "mean_range_without_critical_audio": float(
            without_critical.mean()
        ),
        "p95_range_without_critical_audio": float(
            without_critical.quantile(0.95)
        )
    }

audio = audio.sort_values("pilot_index").reset_index(drop=True)

OUT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUT.with_suffix(".tmp")
audio.to_csv(tmp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(tmp)
    tmp.unlink()
else:
    tmp.replace(OUT)

report = {
    "status": "PASS",
    "scope": "posthoc_diagnostic_no_refitting",
    "unique_audios": 400,
    "hypothesis_rows": 2000,
    "critical_audio": CRITICAL_ID,
    "critical_hypothesis": "H2",
    "critical_true_fluency": float(critical["fluency"]),
    "critical_predictions": {
        name: float(critical[col])
        for name, col in [
            ("A", "acoustic_ridge_prediction"),
            ("B1", "b1_prediction"),
            ("C1", "c1_prediction")
        ]
    },
    "critical_squared_error_shares": sse_share,
    "diagnostics": diagnostics,
    "paired_audio_win_counts": win_counts,
    "stability": stability,
    "paired_audio_csv_sha256": sha256(OUT),
    "source_sha256": {
        path.name: sha256(path)
        for path in EXPECTED
    },
    "interpretation": (
        "No retraining, no primary-analysis exclusions. "
        "MAE comparisons use equal audio weighting. "
        "Speaker-clustered uncertainty not yet estimated."
    )
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, (
        "Existing report differs; original preserved."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_c1_outlier_paired_audit"] = "qc_pass"
progress["step5_c1_outlier_paired_audit_sha256"] = sha256(REPORT)
progress["next_task"] = (
    "Review C1 paired audit and run B2 prompt-aware Ridge"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.7 C1 PAIRED AUDIT SUMMARY ===")
print("Status: PASS")
print("Unique audios:", len(audio))
print("Hypothesis predictions:", len(merged))
print("Critical audio:", CRITICAL_ID)
print("Critical H2 predictions:", report["critical_predictions"])
print("B1 squared-error share (%):", round(100 * sse_share["B1"], 2))
print("C1 squared-error share (%):", round(100 * sse_share["C1"], 2))

print("\n=== DIAGNOSTIC ACCURACY COMPARISON ===")
for condition, models in diagnostics.items():
    print("\n", condition)
    for name, values in models.items():
        print(
            f"{name}: n={values['n']}, "
            f"MAE={values['MAE']:.4f}, "
            f"RMSE={values['RMSE']:.4f}"
        )

print("\n=== PAIRED AUDIO-LEVEL COMPARISON ===")
for name, count in win_counts.items():
    print(f"{name}: {count}")

print("\n=== STABILITY WITH AND WITHOUT CRITICAL AUDIO ===")
for name, values in stability.items():
    print(name)
    for metric, value in values.items():
        print(f"  {metric}: {value:.4f}")

print("\n=== WORST 10 C1 VS A AUDIO DIFFERENCES ===")
print(
    audio.nlargest(
        10, "c1_minus_a_absolute_error"
    )[
        [
            "audio_id", "speaker",
            "a_mae", "c1_mae",
            "c1_minus_a_absolute_error",
            "b1_range", "c1_range"
        ]
    ].round(4).to_string(index=False)
)

print("\nAudio comparison CSV:", OUT)
print("SHA256:", sha256(OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"
B1 = P / "results/step05/b1_transcript_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step05"
PRED = OUTDIR / "b2_prompt_aware_ridge_oof_predictions.csv"
FOLDS = OUTDIR / "b2_prompt_aware_ridge_fold_metrics.csv"
AUDIO = OUTDIR / "b2_prompt_aware_stability_per_audio.csv"
REPORT = P / "reports/step05_b2_prompt_aware_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    B1: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

assert cv["status"] == "frozen"
assert cv["speaker_disjoint"]
assert design["models"]["B2"]["feature_groups"] == [
    "linguistic", "prompt"
]

FEATURES = (
    design["linguistic_feature_names"]
    + design["prompt_feature_names"]
)
assert len(FEATURES) == 9
assert len(set(FEATURES)) == 9

KEYS = ["pilot_index", "audio_id", "speaker",
        "hypothesis_id", "fluency"]
AUDIO_KEYS = ["pilot_index", "audio_id", "speaker", "fluency"]

data = pd.read_csv(
    DATA, dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT, dtype={"audio_id": "string", "speaker": "string"}
)
b1 = pd.read_csv(
    B1, dtype={"audio_id": "string", "speaker": "string"}
)

assert len(data) == len(b1) == 2000
assert len(split) == 400
assert not data.duplicated(KEYS).any()
assert not b1.duplicated(KEYS).any()
assert not split.duplicated(AUDIO_KEYS).any()

df = data.merge(
    split[AUDIO_KEYS + ["validation_fold"]],
    on=AUDIO_KEYS,
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

pred_parts = []
fold_rows = []
ALPHA = 1.0

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold]
    val = df.loc[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert train["speaker"].nunique() == 16
    assert val["speaker"].nunique() == 4
    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    prediction = model.predict(val[FEATURES])
    assert np.isfinite(prediction).all()

    part = val[KEYS + ["validation_fold"]].copy()
    part["b2_prediction"] = prediction
    pred_parts.append(part)

    err = prediction - val["fluency"].to_numpy(dtype=float)
    fold_rows.append({
        "fold": fold,
        "train_audios": 320,
        "validation_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2)))
    })

oof = pd.concat(pred_parts, ignore_index=True).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(oof) == 2000
assert not oof.duplicated(KEYS).any()

comp = oof.merge(
    b1[KEYS + ["validation_fold", "b1_prediction"]],
    on=KEYS + ["validation_fold"],
    how="inner",
    validate="one_to_one"
)
assert len(comp) == 2000

def metrics(frame, col):
    err = frame[col].to_numpy(dtype=float) - frame[
        "fluency"
    ].to_numpy(dtype=float)
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2)))
    }

comparison = {
    "B1": metrics(comp, "b1_prediction"),
    "B2": metrics(comp, "b2_prediction")
}

hypothesis_metrics = {
    h: metrics(group, "b2_prediction")
    for h, group in oof.groupby("hypothesis_id")
}

audio = oof.groupby(
    AUDIO_KEYS + ["validation_fold"], as_index=False
).agg(
    prediction_min=("b2_prediction", "min"),
    prediction_max=("b2_prediction", "max"),
    prediction_sd=("b2_prediction", "std"),
    hypothesis_count=("hypothesis_id", "count")
)

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

stability = {
    "mean_range": float(audio["prediction_range"].mean()),
    "median_range": float(audio["prediction_range"].median()),
    "p95_range": float(audio["prediction_range"].quantile(0.95)),
    "max_range": float(audio["prediction_range"].max()),
    "variable_audio_count": int(
        (audio["prediction_range"] > 1e-10).sum()
    )
}

critical = oof.loc[
    oof["audio_id"].eq("030210164") &
    oof["hypothesis_id"].eq("H2")
]
assert len(critical) == 1

without_critical = comp.loc[
    ~(comp["audio_id"].eq("030210164") &
      comp["hypothesis_id"].eq("H2"))
]

sensitivity = {
    "B1_without_one_H2": metrics(
        without_critical, "b1_prediction"
    ),
    "B2_without_one_H2": metrics(
        without_critical, "b2_prediction"
    )
}

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing result differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv(oof, PRED)
save_csv(pd.DataFrame(fold_rows), FOLDS)
save_csv(audio.sort_values("pilot_index"), AUDIO)

report = {
    "status": "PASS",
    "model": "B2_prompt_aware_ridge",
    "alpha": ALPHA,
    "hyperparameter_tuning": False,
    "features": FEATURES,
    "feature_count": 9,
    "speaker_disjoint_folds": 5,
    "unique_audios": 400,
    "OOF_predictions": 2000,
    "comparison": comparison,
    "per_hypothesis_metrics": hypothesis_metrics,
    "stability": stability,
    "critical_H2_prediction": float(
        critical["b2_prediction"].iloc[0]
    ),
    "sensitivity": sensitivity,
    "source_data_sha256": sha256(DATA),
    "frozen_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED),
    "fold_metrics_sha256": sha256(FOLDS),
    "audio_stability_sha256": sha256(AUDIO),
    "notes": (
        "Fixed-alpha Ridge, no clipping or tuning. "
        "Training-only imputation and scaling. "
        "Prompt disagreement is not ground-truth WER/CER. "
        "All 400 audios remain in primary evaluation."
    )
}

if REPORT.exists():
    old = json.loads(REPORT.read_text(encoding="utf-8"))
    assert old == report, "Existing report differs"
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_b2_prompt_aware_ridge"] = "qc_pass"
progress["step5_b2_predictions_sha256"] = sha256(PRED)
progress["next_task"] = (
    "Review B2 prompt-aware accuracy and instability; "
    "then evaluate C2 prompt-aware fusion"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.8 B2 PROMPT-AWARE RIDGE ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print("OOF predictions:", len(oof))
print("Speaker-disjoint folds: 5")
print("Alpha:", ALPHA)

print("\n=== B1 VS B2 ACCURACY ===")
for name, values in comparison.items():
    print(
        f"{name}: MAE={values['MAE']:.4f}, "
        f"RMSE={values['RMSE']:.4f}"
    )

print("\n=== B2 H1-H5 METRICS ===")
for h, values in sorted(hypothesis_metrics.items()):
    print(
        f"{h}: MAE={values['MAE']:.4f}, "
        f"RMSE={values['RMSE']:.4f}"
    )

print("\n=== B2 PREDICTION STABILITY ===")
for key, value in stability.items():
    print(f"{key}: {value}")

print("\n=== B2 CRITICAL H2 CASE ===")
print("Audio: 030210164")
print("Human fluency:", float(critical["fluency"].iloc[0]))
print("B2 prediction:", float(critical["b2_prediction"].iloc[0]))

print("\n=== SENSITIVITY: WITHOUT ONE H2 PREDICTION ===")
for name, values in sensitivity.items():
    print(
        f"{name}: MAE={values['MAE']:.4f}, "
        f"RMSE={values['RMSE']:.4f}"
    )

print("\n=== B2 PER-FOLD METRICS ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

print("\nPrediction CSV:", PRED)
print("SHA256:", sha256(PRED))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"
C1 = P / "results/step05/c1_fusion_ridge_oof_predictions.csv"
B2 = P / "results/step05/b2_prompt_aware_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step05"
PRED = OUTDIR / "c2_prompt_aware_fusion_ridge_oof_predictions.csv"
FOLDS = OUTDIR / "c2_prompt_aware_fusion_fold_metrics.csv"
AUDIO = OUTDIR / "c2_prompt_aware_fusion_stability_per_audio.csv"
REPORT = P / "reports/step05_c2_prompt_aware_fusion_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    C1: "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    B2: "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.exists(), f"Missing file: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

assert cv["status"] == "frozen"
assert cv["speaker_disjoint"] is True
assert design["models"]["C2"]["feature_groups"] == [
    "acoustic", "linguistic", "prompt"
]

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
c1 = read(C1)
b2 = read(B2)

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL_KEYS = KEYS + ["fluency", "validation_fold"]

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
ACOUSTIC = [
    x for x in data.columns
    if x not in KEYS + ["fluency"] + LING + PROMPT
]
FEATURES = ACOUSTIC + LING + PROMPT

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2
assert len(FEATURES) == 23
assert len(set(FEATURES)) == 23
assert "fluency" not in FEATURES

assert len(data) == 2000
assert len(split) == 400
assert len(c1) == len(b2) == 2000

assert not data.duplicated(KEYS).any()
assert not split.duplicated(IDS).any()

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

ALPHA = 1.0
parts = []
fold_rows = []

for fold in range(1, 6):
    train = df[df["validation_fold"] != fold]
    val = df[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert train["audio_id"].nunique() == 320
    assert val["audio_id"].nunique() == 80
    assert train["speaker"].nunique() == 16
    assert val["speaker"].nunique() == 4

    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    yhat = model.predict(val[FEATURES])
    assert np.isfinite(yhat).all()

    result = val[FULL_KEYS].copy()
    result["c2_prediction"] = yhat
    parts.append(result)

    err = yhat - val["fluency"].to_numpy(dtype=float)

    fold_rows.append({
        "fold": fold,
        "train_speakers": 16,
        "validation_speakers": 4,
        "train_unique_audios": 320,
        "validation_unique_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2)))
    })

oof = pd.concat(parts, ignore_index=True)
oof = oof.sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(oof) == 2000
assert not oof.duplicated(KEYS).any()

# Strictly align with previously stored C1 and B2 predictions.
comparison = oof.merge(
    c1[FULL_KEYS + ["c1_prediction"]],
    on=FULL_KEYS,
    how="inner",
    validate="one_to_one"
).merge(
    b2[FULL_KEYS + ["b2_prediction"]],
    on=FULL_KEYS,
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 2000

def metrics(frame, prediction_column):
    error = (
        frame[prediction_column].to_numpy(dtype=float)
        - frame["fluency"].to_numpy(dtype=float)
    )
    return {
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2)))
    }

model_comparison = {
    "B2": metrics(comparison, "b2_prediction"),
    "C1": metrics(comparison, "c1_prediction"),
    "C2": metrics(comparison, "c2_prediction")
}

hypothesis_metrics = {
    hid: metrics(group, "c2_prediction")
    for hid, group in oof.groupby("hypothesis_id")
}

audio = oof.groupby(
    IDS + ["fluency", "validation_fold"],
    as_index=False
).agg(
    prediction_min=("c2_prediction", "min"),
    prediction_max=("c2_prediction", "max"),
    prediction_sd=("c2_prediction", "std"),
    prediction_mean=("c2_prediction", "mean"),
    hypothesis_count=("hypothesis_id", "count")
)

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

stability = {
    "mean_range": float(audio["prediction_range"].mean()),
    "median_range": float(audio["prediction_range"].median()),
    "p95_range": float(audio["prediction_range"].quantile(0.95)),
    "max_range": float(audio["prediction_range"].max()),
    "mean_prediction_sd": float(audio["prediction_sd"].mean()),
    "variable_audio_count": int(
        (audio["prediction_range"] > 1e-10).sum()
    )
}

CRITICAL_ID = "030210164"

critical_mask = (
    comparison["audio_id"].eq(CRITICAL_ID)
    & comparison["hypothesis_id"].eq("H2")
)

assert critical_mask.sum() == 1
critical = comparison.loc[critical_mask].iloc[0]

without_one = comparison.loc[~critical_mask]
without_audio = comparison[
    comparison["audio_id"] != CRITICAL_ID
]

sensitivity = {
    "without_one_H2": {
        name: metrics(without_one, f"{name.lower()}_prediction")
        for name in ["B2", "C1", "C2"]
    },
    "without_entire_audio": {
        name: metrics(without_audio, f"{name.lower()}_prediction")
        for name in ["B2", "C1", "C2"]
    }
}

c2_error = (
    comparison["c2_prediction"] - comparison["fluency"]
)
c2_critical_sse_share = float(
    (critical["c2_prediction"] - critical["fluency"]) ** 2
    / np.square(c2_error).sum()
)

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")

    if path.exists():
        assert sha256(path) == sha256(temp), (
            f"Existing result differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(oof, PRED)
save_csv(pd.DataFrame(fold_rows), FOLDS)
save_csv(audio.sort_values("pilot_index"), AUDIO)

report = {
    "status": "PASS",
    "model": "C2_acoustic_linguistic_prompt_aware_ridge",
    "feature_count": 23,
    "acoustic_features": 14,
    "linguistic_features": 7,
    "prompt_features": 2,
    "alpha": ALPHA,
    "hyperparameter_tuning_performed": False,
    "prediction_clipping_performed": False,
    "speaker_disjoint_cv": "frozen_5fold",
    "unique_audio_count": 400,
    "OOF_predictions": 2000,
    "comparison": model_comparison,
    "per_hypothesis_metrics": hypothesis_metrics,
    "stability": stability,
    "critical_audio_H2": {
        "human_fluency": float(critical["fluency"]),
        "B2_prediction": float(critical["b2_prediction"]),
        "C1_prediction": float(critical["c1_prediction"]),
        "C2_prediction": float(critical["c2_prediction"]),
        "C2_squared_error_share": c2_critical_sse_share
    },
    "posthoc_sensitivity": sensitivity,
    "source_dataset_sha256": sha256(DATA),
    "frozen_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED),
    "fold_metrics_sha256": sha256(FOLDS),
    "audio_stability_sha256": sha256(AUDIO),
    "notes": (
        "Unclipped fixed-alpha Ridge predictions. "
        "Train-fold-only median imputation and scaling. "
        "Each audio contributes exactly five hypotheses. "
        "Prompt-ASR disagreement is not verified WER/CER. "
        "No primary-analysis observations excluded."
    )
}

if REPORT.exists():
    saved = json.loads(REPORT.read_text(encoding="utf-8"))
    assert saved == report, (
        "Existing C2 report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_c2_prompt_aware_fusion"] = "qc_pass"
progress["step5_c2_predictions_sha256"] = sha256(PRED)
progress["next_task"] = (
    "Compare five fixed-alpha models and estimate "
    "speaker-clustered uncertainty for paired differences"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.9 C2 PROMPT-AWARE FUSION ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print("Unique audios:", 400)
print("OOF predictions:", len(oof))
print("Speaker-disjoint CV: PASS")
print("Ridge alpha:", ALPHA)

print("\n=== B2 VS C1 VS C2 ACCURACY ===")
for name, m in model_comparison.items():
    print(
        f"{name}: MAE={m['MAE']:.4f}, "
        f"RMSE={m['RMSE']:.4f}"
    )

print("\n=== C2 H1-H5 METRICS ===")
for hid, m in sorted(hypothesis_metrics.items()):
    print(
        f"{hid}: MAE={m['MAE']:.4f}, "
        f"RMSE={m['RMSE']:.4f}"
    )

print("\n=== C2 PREDICTION STABILITY ===")
for key, value in stability.items():
    print(f"{key}: {value:.4f}")

print("\n=== C2 CRITICAL H2 CASE ===")
for key, value in report["critical_audio_H2"].items():
    print(f"{key}: {value}")

print("\n=== SENSITIVITY WITHOUT ONE H2 ===")
for name, m in sensitivity["without_one_H2"].items():
    print(
        f"{name}: MAE={m['MAE']:.4f}, "
        f"RMSE={m['RMSE']:.4f}"
    )

print("\n=== C2 PER-FOLD METRICS ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

print("\nPrediction CSV:", PRED)
print("SHA256:", sha256(PRED))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
DIR = P / "results/step05"
CHECK = P / "checkpoints/research_progress.json"

SOURCES = {
    "A": DIR / "baseline_acoustic_oof_predictions.csv",
    "B1": DIR / "b1_transcript_ridge_oof_predictions.csv",
    "B2": DIR / "b2_prompt_aware_ridge_oof_predictions.csv",
    "C1": DIR / "c1_fusion_ridge_oof_predictions.csv",
    "C2": DIR / "c2_prompt_aware_fusion_ridge_oof_predictions.csv"
}

EXPECTED = {
    "A": "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    "B1": "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    "B2": "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    "C1": "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    "C2": "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

PRED_COLUMNS = {
    "A": "acoustic_ridge_prediction",
    "B1": "b1_prediction",
    "B2": "b2_prediction",
    "C1": "c1_prediction",
    "C2": "c2_prediction"
}

OUT = DIR / "speaker_cluster_bootstrap_10000_ci.csv"
REPORT = P / "reports/step06_speaker_cluster_bootstrap_v1.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for model, path in SOURCES.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == EXPECTED[model], (
        f"SHA mismatch for {model}"
    )

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

frames = {name: read(path) for name, path in SOURCES.items()}

KEYS = [
    "pilot_index", "audio_id", "speaker",
    "fluency", "validation_fold"
]
HKEYS = KEYS + ["hypothesis_id"]

assert len(frames["A"]) == 400

for name in ["B1", "B2", "C1", "C2"]:
    assert len(frames[name]) == 2000
    assert not frames[name].duplicated(HKEYS).any()

base = frames["B1"][
    HKEYS + [PRED_COLUMNS["B1"]]
].copy()

for name in ["B2", "C1", "C2"]:
    base = base.merge(
        frames[name][HKEYS + [PRED_COLUMNS[name]]],
        on=HKEYS,
        how="inner",
        validate="one_to_one"
    )

base = base.merge(
    frames["A"][KEYS + [PRED_COLUMNS["A"]]],
    on=KEYS,
    how="inner",
    validate="many_to_one"
)

assert len(base) == 2000
assert base["audio_id"].nunique() == 400
assert base["speaker"].nunique() == 20
assert base.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert base.groupby("speaker")["validation_fold"].nunique().eq(1).all()
assert base.groupby("speaker")["audio_id"].nunique().eq(20).all()

# One row per audio: average the five hypothesis errors
# BEFORE speaker-level resampling.


# Explicit error columns avoid ambiguity in aggregation.
for name, col in PRED_COLUMNS.items():
    base[f"{name}_ae"] = (
        base[col] - base["fluency"]
    ).abs()
    base[f"{name}_se"] = (
        base[col] - base["fluency"]
    ) ** 2

agg = {}
for name in PRED_COLUMNS:
    agg[f"{name}_ae"] = (f"{name}_ae", "mean")
    agg[f"{name}_se"] = (f"{name}_se", "mean")
    agg[f"{name}_min"] = (PRED_COLUMNS[name], "min")
    agg[f"{name}_max"] = (PRED_COLUMNS[name], "max")

audio = base.groupby(
    KEYS, as_index=False
).agg(**agg)

assert len(audio) == 400
assert audio["audio_id"].is_unique

for name in PRED_COLUMNS:
    audio[f"{name}_range"] = (
        audio[f"{name}_max"] - audio[f"{name}_min"]
    )

# A must be invariant to ASR hypotheses.
assert np.allclose(audio["A_range"], 0, atol=1e-12)

MODELS = ["A", "B1", "B2", "C1", "C2"]
SPEAKERS = sorted(audio["speaker"].unique())
assert len(SPEAKERS) == 20

# Each speaker has 20 audios, so equal speaker weights
# are also equal audio weights in this pilot.
speaker = audio.groupby("speaker").agg(
    **{
        f"{name}_{metric}": (f"{name}_{metric}", "mean")
        for name in MODELS
        for metric in ["ae", "se", "range"]
    }
).reindex(SPEAKERS)

assert speaker.shape[0] == 20
assert speaker.notna().all().all()

N_BOOT = 10000
SEED = 20261026
rng = np.random.default_rng(SEED)

# Matrix: iterations x sampled speakers.
indices = rng.integers(
    low=0,
    high=20,
    size=(N_BOOT, 20)
)

bootstrap = {}
observed = {}

for name in MODELS:
    for metric in ["ae", "se", "range"]:
        values = speaker[f"{name}_{metric}"].to_numpy(
            dtype=float
        )

        avg = values[indices].mean(axis=1)

        if metric == "se":
            bootstrap[(name, "RMSE")] = np.sqrt(avg)
            observed[(name, "RMSE")] = float(
                np.sqrt(values.mean())
            )
        elif metric == "ae":
            bootstrap[(name, "MAE")] = avg
            observed[(name, "MAE")] = float(values.mean())
        else:
            bootstrap[(name, "mean_range")] = avg
            observed[(name, "mean_range")] = float(
                values.mean()
            )

# Verify observed metrics agree with previous reports.
expected_metrics = {
    "A": (0.9399, 1.2066),
    "B1": (1.2426, 1.9972),
    "B2": (1.1761, 1.9518),
    "C1": (0.8972, 1.3735),
    "C2": (0.7782, 1.2461)
}

for name, (mae, rmse) in expected_metrics.items():
    assert np.isclose(observed[(name, "MAE")], mae, atol=0.0001)
    assert np.isclose(observed[(name, "RMSE")], rmse, atol=0.0001)

def ci(values):
    return [
        float(np.quantile(values, 0.025)),
        float(np.quantile(values, 0.975))
    ]

rows = []

for name in MODELS:
    for metric in ["MAE", "RMSE", "mean_range"]:
        low, high = ci(bootstrap[(name, metric)])
        rows.append({
            "comparison": name,
            "metric": metric,
            "observed": observed[(name, metric)],
            "ci_2_5": low,
            "ci_97_5": high
        })

# Paired differences are calculated within the SAME
# bootstrap speaker draws for both models.
PAIRINGS = [
    ("C1", "A"),
    ("C2", "A"),
    ("C2", "C1"),
    ("B2", "B1"),
    ("C1", "B1"),
    ("C2", "B2")
]

for model, reference in PAIRINGS:
    for metric in ["MAE", "RMSE", "mean_range"]:
        diff = (
            bootstrap[(model, metric)]
            - bootstrap[(reference, metric)]
        )
        low, high = ci(diff)

        rows.append({
            "comparison": f"{model}_minus_{reference}",
            "metric": metric,
            "observed": (
                observed[(model, metric)]
                - observed[(reference, metric)]
            ),
            "ci_2_5": low,
            "ci_97_5": high
        })

results = pd.DataFrame(rows)

assert len(results) == 33
assert results[["observed", "ci_2_5", "ci_97_5"]].notna().all().all()

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUT.with_suffix(".tmp")
results.to_csv(tmp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(tmp), (
        "Existing bootstrap CSV differs; preserved."
    )
    tmp.unlink()
else:
    tmp.replace(OUT)

report = {
    "status": "PASS",
    "method": "paired_speaker_cluster_percentile_bootstrap",
    "bootstrap_iterations": N_BOOT,
    "random_seed": SEED,
    "confidence_level": 0.95,
    "resampling_unit": "speaker",
    "unique_speakers": 20,
    "unique_audios": 400,
    "hypotheses_per_audio": 5,
    "source_split": "frozen_5fold_speaker_disjoint",
    "all_oof_predictions_unchanged": True,
    "all_outliers_retained": True,
    "refitting_during_bootstrap": False,
    "models": MODELS,
    "comparison_pairs": [
        f"{m}_minus_{r}" for m, r in PAIRINGS
    ],
    "results_sha256": sha256(OUT),
    "source_sha256": {
        name: sha256(path)
        for name, path in SOURCES.items()
    },
    "interpretation": (
        "Paired speaker-cluster bootstrap CIs conditional "
        "on previously fitted OOF models; not nested-CV "
        "or refitting uncertainty. "
        "Negative error difference favors the first model. "
        "Bootstrap intervals are descriptive, especially "
        "with only 20 speaker clusters."
    )
}

if REPORT.exists():
    old = json.loads(REPORT.read_text(encoding="utf-8"))
    assert old == report, (
        "Existing bootstrap report differs; preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_speaker_cluster_bootstrap"] = "qc_pass"
progress["step6_bootstrap_results_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Review speaker-cluster bootstrap uncertainty; "
    "perform robustness and final model comparison"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.1 SPEAKER-CLUSTER BOOTSTRAP ===")
print("Status: PASS")
print("Bootstrap iterations:", N_BOOT)
print("Speaker clusters:", len(SPEAKERS))
print("Confidence interval: 95%")
print("Refitting: NO")
print("All 400 audios retained: YES")

print("\n=== MODEL ACCURACY AND 95% CI ===")
print(
    results[
        results["comparison"].isin(MODELS) &
        results["metric"].isin(["MAE", "RMSE"])
    ].round(4).to_string(index=False)
)

print("\n=== PAIRED MODEL DIFFERENCES AND 95% CI ===")
print(
    results[
        results["comparison"].str.contains("_minus_") &
        results["metric"].isin(["MAE", "RMSE"])
    ].round(4).to_string(index=False)
)

print("\n=== PREDICTION STABILITY AND 95% CI ===")
print(
    results[results["metric"] == "mean_range"]
    .round(4).to_string(index=False)
)

print("\nCSV:", OUT)
print("SHA256:", sha256(OUT))
print("Report:", REPORT)
print("Checkpoint updated.")
